# KissanSevaAI — Task 3: Full-Dataset FAISS Index, Classifier Benchmark & OOD Uncertainty Calibration

**Objectives:**
1. Verify actual archive structure in `/content/drive/MyDrive/archive.zip` (`/content/plant_disease_dataset/crops/<crop>/<disease>/`) and audit class alignment against the existing **67-class** production map.
2. Reuse existing split manifest if present, OR generate a leakage-controlled deterministic **80/10/10** (`train`/`val`/`test`) split with exact **SHA256** deduplication + perceptual **dHash** clustering + post-embedding visual similarity audit.
3. Extract full **DINOv2-S/14** (`CLS + mean patch`, 768-D L2-normalized) training embeddings using exact ONNX preprocessing (`Short=256 Bilinear -> CenterCrop 224 -> ImageNet Mean/Std`) with resumable atomic chunks.
4. Build a separate versioned **`faiss.IndexFlatIP(768)`** from **training images only** (never overwriting production `v1` assets).
5. Benchmark 7 candidate methods on the exact **same validation split** (`k=1`, `k=3/5/7/11/21`, similarity-weighted voting, class-balanced retrieval, crop-aware retrieval, linear classifier, SVM) and compare the validation-selected winner **once** against the old `2,144`-reference system on the untouched test split.
6. Calibrate explicit uncertainty & OOD rejection (`unknown` / `needs_review`) on validation data across blurry, dark/poor-lighting, unsupported crops, ambiguous diseases, and non-leaf photographs (animals, machinery, faces, non-plant backgrounds).


## 1. Environment Setup, Drive Mount & Production Safety Guard


In [ ]:
!pip -q install faiss-cpu scikit-learn pandas pillow tqdm onnx onnxruntime opencv-python-headless scipy

import hashlib
import json
import os
import random
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import cv2
import faiss
import numpy as np
import onnxruntime as ort
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets as tv_datasets, transforms
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Enable GPU in Colab: Runtime -> Change runtime type -> T4 / L4 GPU'
DEVICE = 'cuda'
print('Running on GPU:', torch.cuda.get_device_name(0))

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Paths
ZIP_PATH = Path('/content/drive/MyDrive/archive.zip')
WORK_DIR = Path('/content/plant_disease_dataset')
VERSION_TAG = 'v3_task3_full_faiss'
OUT_DIR = Path('/content/drive/MyDrive/KissanSevaAI_Phase1/versions') / VERSION_TAG
ASSETS_OUT = OUT_DIR / 'model_assets'
EVAL_OUT = OUT_DIR / 'evaluation'
CHUNKS_DIR = OUT_DIR / 'embedding_chunks'

# Optional paths to compare directly against the existing 2,144-reference v1 index and reuse v1 splits if present
V1_ASSETS_DIR = Path('/content/drive/MyDrive/KissanSevaAI_Phase1/pilot')  # Upload ml-service/model_assets here if testing v1 vs v2 side-by-side
EXISTING_SPLIT_CANDIDATES = [
    Path('/content/drive/MyDrive/KissanSevaAI_Phase1/full/splits.csv'),
    OUT_DIR / 'splits_manifest.csv',
]

for d in [OUT_DIR, ASSETS_OUT, EVAL_OUT, CHUNKS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.is_file() and zipfile.is_zipfile(ZIP_PATH), f'Dataset ZIP not found: {ZIP_PATH}'
assert ASSETS_OUT.resolve() != V1_ASSETS_DIR.resolve(), 'Safety Guard: Versioned output must NOT overwrite v1 production folder!'

SIZE = 224
RESIZE_SHORT = 256
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
BATCH_SIZE = 32
CHUNK_SIZE = 1024


## 2. Extract Dataset & Verify Structure Against Existing 67-Class Map


In [ ]:
CANONICAL_CLASSES = [
  "Apple/Apple_Black_rot",
  "Apple/Apple_Cedar_apple_rust",
  "Apple/Apple_healthy",
  "Apple/Apple_scab",
  "Cherry/Cherry_Healthy",
  "Cherry/Cherry_Powdery_mildew",
  "Citrus/Citrus_Black_spot",
  "Citrus/Citrus_Healthy",
  "Citrus/Citrus_canker",
  "Citrus/Citrus_greening",
  "Corn/Corn_Common_rust",
  "Corn/Corn_Northern_Leaf_Blight",
  "Corn/Corn_cercospora_gray_leaf_spot",
  "Corn/Corn_healthy",
  "Cotton/Cotton_bacterial_blight",
  "Cotton/Cotton_curl_virus",
  "Cotton/Cotton_fussarium_wilt",
  "Cotton/Cotton_healthy",
  "Grape/Grape_Black_Measles",
  "Grape/Grape_Black_rot",
  "Grape/Grape_Healthy",
  "Grape/Grape_Isariopsis_Leaf_Spot",
  "Peach/Peach_Bacterial_spot",
  "Peach/Peach_Healthy",
  "Pepper_bell/Pepper_bell_Bacterial_spot",
  "Pepper_bell/pepper_bell_healthy",
  "Potato/Potato_Early_blight",
  "Potato/Potato_Late_blight",
  "Potato/Potato_healthy",
  "Rice/Rice_Bacterial_Leaf_Blight",
  "Rice/Rice_Brown_Spot",
  "Rice/Rice_Healthy",
  "Rice/Rice_Leaf_Blast",
  "Rice/Rice_Leaf_scald",
  "Rice/Rice_Sheath_Blight",
  "Strawberry/Strawberry_Healthy",
  "Strawberry/Strawberry_Leaf_scorch",
  "Sugarcane/Sugarcane_Healthy",
  "Sugarcane/Sugarcane_Mosaic_Virus",
  "Sugarcane/Sugarcane_RedRot",
  "Sugarcane/Sugarcane_Rust",
  "Sugarcane/Sugarcane_Yellow",
  "Tomato/Tomato_Bacterial_spot",
  "Tomato/Tomato_Early_blight",
  "Tomato/Tomato_Late_blight",
  "Tomato/Tomato_Leaf_Mold",
  "Tomato/Tomato_Septoria_leaf_spot",
  "Tomato/Tomato_Spider_mites",
  "Tomato/Tomato_Target_Spot",
  "Tomato/Tomato_YellowLeaf_Curl_Virus",
  "Tomato/Tomato_healthy",
  "Tomato/Tomato_mosaic_virus",
  "Wheat/Wheat_Aphid",
  "Wheat/Wheat_Black_Rust",
  "Wheat/Wheat_Blast",
  "Wheat/Wheat_Brown_Rust",
  "Wheat/Wheat_Common_Root_Rot",
  "Wheat/Wheat_Fusarium_Head_Blight",
  "Wheat/Wheat_Healthy",
  "Wheat/Wheat_Leaf_Blight",
  "Wheat/Wheat_Mildew",
  "Wheat/Wheat_Mite",
  "Wheat/Wheat_Septoria",
  "Wheat/Wheat_Smut",
  "Wheat/Wheat_Stem_fly",
  "Wheat/Wheat_Tan_spot",
  "Wheat/Wheat_Yellow_Rust"
]
CLASS_TO_ID = {c: i for i, c in enumerate(CANONICAL_CLASSES)}
NORM_LOOKUP = {c.lower().replace(' ', '_').replace('___', '/'): c for c in CANONICAL_CLASSES}
(ASSETS_OUT / 'classes.json').write_text(json.dumps(CANONICAL_CLASSES, indent=2))

DONE_FLAG = WORK_DIR / '.extract_done'
if not DONE_FLAG.exists():
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        base = WORK_DIR.resolve()
        for member in z.infolist():
            target = (base / member.filename).resolve()
            if not (base == target or base in target.parents):
                raise ValueError(f'Unsafe ZIP path detected: {member.filename}')
        z.extractall(WORK_DIR)
    DONE_FLAG.touch()

# Locate /content/plant_disease_dataset/crops/
preferred = WORK_DIR / 'crops'
if preferred.is_dir():
    CROPS_DIR = preferred
else:
    roots = [p for p in WORK_DIR.rglob('crops') if p.is_dir()]
    assert roots, f'Cannot find crops/ folder inside {WORK_DIR}'
    CROPS_DIR = min(roots, key=lambda p: len(p.parts))

print('Verified CROPS_DIR:', CROPS_DIR)

# Audit every <crop>/<disease>/ folder inside archive without assuming 67-class match
EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
discovered_dirs = {}
for crop_dir in sorted(CROPS_DIR.iterdir()):
    if not crop_dir.is_dir():
        continue
    for cond_dir in sorted(crop_dir.iterdir()):
        if not cond_dir.is_dir():
            continue
        raw_label = f'{crop_dir.name}/{cond_dir.name}'
        imgs = [p for p in sorted(cond_dir.rglob('*')) if p.is_file() and p.suffix.lower() in EXTS]
        discovered_dirs[raw_label] = imgs

matched_exact = [k for k in discovered_dirs if k in CLASS_TO_ID]
norm_matched = {}
unsupported_classes = {}
for k, imgs in discovered_dirs.items():
    if k in CLASS_TO_ID:
        continue
    nk = k.lower().replace(' ', '_').replace('___', '/')
    if nk in NORM_LOOKUP:
        norm_matched[k] = NORM_LOOKUP[nk]
    else:
        unsupported_classes[k] = len(imgs)

missing_canonical = [c for c in CANONICAL_CLASSES if c not in discovered_dirs and c not in norm_matched.values()]

audit_summary = {
    'total_discovered_folders': len(discovered_dirs),
    'total_raw_images': sum(len(v) for v in discovered_dirs.values()),
    'matched_exact_67': len(matched_exact),
    'matched_via_normalization': norm_matched,
    'unsupported_extra_classes': unsupported_classes,
    'missing_from_archive': missing_canonical,
    'sample_filenames': [str(p.relative_to(CROPS_DIR)) for k in matched_exact[:5] for p in discovered_dirs[k][:2]],
}
(EVAL_OUT / 'archive_structure_audit.json').write_text(json.dumps(audit_summary, indent=2))
print(json.dumps({k: v for k, v in audit_summary.items() if k != 'unsupported_extra_classes'}, indent=2))
print(f'Extra classes outside 67-class map (saved for Unsupported-Crop OOD testing): {len(unsupported_classes)}')
assert len(missing_canonical) == 0, f'Missing canonical classes in archive: {missing_canonical}' 


## 3. Reproducible Leakage-Controlled Split & Near-Duplicate Investigation

1. Computes **SHA256**, **64-bit perceptual dHash**, image dimensions, Laplacian blur score, and brightness for every eligible image.
2. Saves extra unsupported-crop images separately (`unsupported_crops_manifest.csv`) for Task 7 OOD evaluation.
3. Reuses an existing split manifest if found (`EXISTING_SPLIT_CANDIDATES`), enforcing that NO validation/test image enters `train` and fixing any cross-split near-duplicates. Otherwise performs deterministic **`StratifiedGroupKFold(10)`** grouped by perceptual near-duplicate clusters `(canonical_label, dhash_bucket)`.


In [ ]:
SPLITS_CSV = OUT_DIR / 'splits_manifest.csv'
UNSUPPORTED_CSV = OUT_DIR / 'unsupported_crops_manifest.csv'

def compute_dhash_and_quality(img_path: Path, hash_size: int = 8):
    with Image.open(img_path) as raw:
        raw.load()
        w, h = raw.size
        if w < 32 or h < 32 or (w * h) > 40_000_000:
            raise ValueError(f'Invalid dimensions {w}x{h}')
        im_rgb = ImageOps.exif_transpose(raw).convert('RGB')
        gray_small = im_rgb.convert('L').resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
        arr_h = np.asarray(gray_small, dtype=np.int16)
        diff = arr_h[:, 1:] > arr_h[:, :-1]
        bits = 0
        for b in diff.flatten():
            bits = (bits << 1) | int(bool(b))
        dhash_hex = f'{bits:016x}'
        # Blur (Laplacian variance) & mean brightness on standard 256px view
        arr_gray = np.asarray(im_rgb.convert('L').resize((256, 256)), dtype=np.uint8)
        blur_var = float(cv2.Laplacian(arr_gray, cv2.CV_64F).var())
        brightness = float(arr_gray.mean())
        return w, h, dhash_hex, blur_var, brightness

def hamming_hex(a: str, b: str) -> int:
    return bin(int(a, 16) ^ int(b, 16)).count('1')

if SPLITS_CSV.exists():
    split_df = pd.read_csv(SPLITS_CSV)
    print('Reused saved splits_manifest.csv:', len(split_df))
else:
    eligible_rows = []
    unsupported_rows = []
    seen_sha = {}
    conflict_shas = set()
    corrupt_files = []
    exact_dupe_count = 0

    all_items = []
    for raw_label, imgs in discovered_dirs.items():
        if raw_label in CLASS_TO_ID:
            canon = raw_label
        elif raw_label in norm_matched:
            canon = norm_matched[raw_label]
        else:
            canon = None
        for p in imgs:
            all_items.append((raw_label, canon, p))

    for raw_label, canon, img_path in tqdm(all_items, desc='Hashing & Quality Audit'):
        rel_path = str(img_path.relative_to(CROPS_DIR))
        sha = hashlib.sha256(img_path.read_bytes()).hexdigest()
        if canon is None:
            unsupported_rows.append({'path': rel_path, 'raw_label': raw_label, 'sha256': sha})
            continue
        if sha in seen_sha:
            if seen_sha[sha] != canon:
                conflict_shas.add(sha)
            exact_dupe_count += 1
            continue
        seen_sha[sha] = canon
        try:
            w, h, dhash_hex, blur_var, brightness = compute_dhash_and_quality(img_path)
        except Exception as exc:
            corrupt_files.append({'path': rel_path, 'error': str(exc)})
            continue
        eligible_rows.append({
            'path': rel_path,
            'raw_label': raw_label,
            'label': canon,
            'crop': canon.split('/')[0],
            'class_id': CLASS_TO_ID[canon],
            'sha256': sha,
            'dhash': dhash_hex,
            'width': w,
            'height': h,
            'blur_laplacian_var': blur_var,
            'brightness': brightness,
        })

    df = pd.DataFrame(eligible_rows)
    if conflict_shas:
        df = df[~df.sha256.isin(conflict_shas)].reset_index(drop=True)
    pd.DataFrame(unsupported_rows).to_csv(UNSUPPORTED_CSV, index=False)

    # Check filename clues for augmentation duplicates (e.g. rotated/flipped variants)
    df['stem_lower'] = df['path'].apply(lambda s: Path(s).stem.lower())
    aug_pattern_hits = int(df['stem_lower'].str.contains(r'(?:_aug|_rot|_flip|copy|clone|_90|_180|_270)', regex=True).sum())

    # Cluster near-identical perceptual hashes within each class (Hamming distance <= 4)
    cluster_ids = np.arange(len(df), dtype=np.int32)
    for cid, grp in df.groupby('class_id'):
        idxs = grp.index.to_numpy()
        hashes = [int(h, 16) for h in grp['dhash']]
        exact_map = {}
        for pos, (row_idx, hv) in enumerate(zip(idxs, hashes)):
            # Coarse bucket by top 48 bits catches slight crops/recompression; check exact first
            key48 = hv >> 16
            if key48 in exact_map:
                cluster_ids[row_idx] = exact_map[key48]
            else:
                exact_map[key48] = int(row_idx)
    df['near_dup_cluster'] = df['label'] + '::' + pd.Series(cluster_ids).astype(str)

    # Reuse original Colab split manifest if available on Drive
    reused_orig = None
    for cand in EXISTING_SPLIT_CANDIDATES:
        if cand.exists() and cand != SPLITS_CSV:
            tmp_orig = pd.read_csv(cand)
            if {'sha256', 'split'}.issubset(tmp_orig.columns):
                reused_orig = tmp_orig[['sha256', 'split']].drop_duplicates('sha256')
                print('Found existing Colab split manifest to preserve:', cand)
                break

    if reused_orig is not None:
        sha_to_split = dict(zip(reused_orig.sha256, reused_orig.split))
        df['split'] = df['sha256'].map(sha_to_split).fillna('train')
        # Force any same-cluster member sharing a near-duplicate cluster with val/test OUT of train to eliminate leakage!
        non_train_clusters = set(df[df.split != 'train'].near_dup_cluster)
        df.loc[(df.split == 'train') & (df.near_dup_cluster.isin(non_train_clusters)), 'split'] = 'purged_near_dup_leakage'
    else:
        sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
        fold_ids = np.zeros(len(df), dtype=np.int32)
        for fold, (_, ti) in enumerate(sgkf.split(df, df['class_id'], df['near_dup_cluster'])):
            fold_ids[ti] = fold
        df['split'] = np.where(fold_ids == 9, 'test', np.where(fold_ids == 8, 'val', 'train'))

    df = df.sort_values(['split', 'class_id', 'path']).reset_index(drop=True)
    df.insert(0, 'image_id', [f'img_{i:06d}' for i in range(len(df))])
    df.to_csv(SPLITS_CSV, index=False)
    split_df = df

    dup_stats = {
        'total_eligible_unique_sha': len(split_df),
        'dropped_exact_sha_dupes': exact_dupe_count,
        'dropped_conflicting_sha': len(conflict_shas),
        'dropped_corrupt_or_tiny': len(corrupt_files),
        'near_dup_clusters': int(split_df['near_dup_cluster'].nunique()),
        'filename_aug_pattern_hits': aug_pattern_hits,
        'unsupported_crop_images': len(unsupported_rows),
        'split_counts': split_df['split'].value_counts().to_dict(),
    }
    (EVAL_OUT / 'deduplication_and_split_audit.json').write_text(json.dumps(dup_stats, indent=2))
    print(json.dumps(dup_stats, indent=2))

train_df = split_df[split_df.split == 'train'].reset_index(drop=True)
val_df = split_df[split_df.split == 'val'].reset_index(drop=True)
test_df = split_df[split_df.split == 'test'].reset_index(drop=True)
assert set(train_df.sha256).isdisjoint(val_df.sha256)
assert set(train_df.sha256).isdisjoint(test_df.sha256)
assert set(train_df.near_dup_cluster).isdisjoint(val_df.near_dup_cluster)
assert set(train_df.near_dup_cluster).isdisjoint(test_df.near_dup_cluster)
print(f'Verified Split Counts -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)} | Leakage = 0 clusters')


## 4. Exact DINOv2-S/14 Weights, Preprocessing & Resumable Chunked Embedding Extraction


In [ ]:
PREPROCESS = transforms.Compose([
    transforms.Resize(RESIZE_SHORT, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', trust_repo=True).eval().to(DEVICE)
backbone.requires_grad_(False)

class DinoFeatures(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, x):
        feat = self.encoder.forward_features(x)
        cls = feat['x_norm_clstoken']
        avg = feat['x_norm_patchtokens'].mean(dim=1)
        return F.normalize(torch.cat([cls, avg], dim=1), dim=1)

model = DinoFeatures(backbone).eval().to(DEVICE)

# Export and verify ONNX parity (and verify against v1 ONNX if present on Drive)
onnx_out = ASSETS_OUT / 'dinov2_vits14_cls_mean.onnx'
dummy = torch.randn(1, 3, SIZE, SIZE, device=DEVICE)
with torch.inference_mode():
    torch.onnx.export(
        model, dummy, str(onnx_out),
        input_names=['image'], output_names=['embedding'],
        dynamic_axes={'image': {0: 'batch'}, 'embedding': {0: 'batch'}},
        opset_version=17, do_constant_folding=True, dynamo=False
    )
    pt_ref = model(dummy).float().cpu().numpy()

ort_sess = ort.InferenceSession(str(onnx_out), providers=['CPUExecutionProvider'])
onx_ref = ort_sess.run(None, {'image': dummy.detach().cpu().numpy()})[0]
assert float(np.abs(pt_ref - onx_ref).max()) < 0.01, 'ONNX export differs from PyTorch'

v1_onnx = V1_ASSETS_DIR / 'dinov2_vits14_cls_mean.onnx'
if v1_onnx.exists():
    v1_sess = ort.InferenceSession(str(v1_onnx), providers=['CPUExecutionProvider'])
    v1_out = v1_sess.run(None, {'image': dummy.detach().cpu().numpy()})[0]
    cos_v1 = float(np.dot(onx_ref[0], v1_out[0]))
    print(f'Verified New ONNX <-> v1 ONNX cosine parity: {cos_v1:.6f}')
    assert cos_v1 > 0.9999

class ImgDataset(Dataset):
    def __init__(self, frame):
        self.paths = [CROPS_DIR / p for p in frame['path']]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as raw:
            return PREPROCESS(ImageOps.exif_transpose(raw).convert('RGB'))

def extract_chunked(split_name: str, frame: pd.DataFrame):
    assert frame['sha256'].is_unique, f'Duplicate SHA256 inside {split_name}'
    sdir = CHUNKS_DIR / split_name
    sdir.mkdir(parents=True, exist_ok=True)
    chunks = []
    for start in tqdm(range(0, len(frame), CHUNK_SIZE), desc=f'Extract {split_name}'):
        stop = min(start + CHUNK_SIZE, len(frame))
        cfile = sdir / f'chunk_{start:07d}_{stop:07d}.npy'
        mfile = sdir / f'chunk_{start:07d}_{stop:07d}.json'
        ids_expected = frame.iloc[start:stop]['sha256'].tolist()
        valid_cache = False
        if cfile.exists() and mfile.exists():
            if json.loads(mfile.read_text()).get('sha256') == ids_expected:
                arr = np.load(cfile)
                if arr.shape == (stop - start, 768) and np.isfinite(arr).all():
                    valid_cache = True
        if not valid_cache:
            dl = DataLoader(ImgDataset(frame.iloc[start:stop]), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
            out_list = []
            with torch.inference_mode():
                for xb in dl:
                    with torch.autocast('cuda', dtype=torch.float16):
                        vec = model(xb.to(DEVICE, non_blocking=True)).float().cpu().numpy()
                    out_list.append(vec)
            arr = np.concatenate(out_list, axis=0).astype('float32')
            arr /= np.maximum(np.linalg.norm(arr, axis=1, keepdims=True), 1e-9)
            tmp = cfile.with_suffix('.tmp.npy')
            np.save(tmp, arr)
            os.replace(tmp, cfile)
            mfile.write_text(json.dumps({'sha256': ids_expected}))
        chunks.append(arr)
    X = np.concatenate(chunks, axis=0).astype('float32')
    y = frame['class_id'].to_numpy(dtype=np.int32)
    assert np.allclose(np.linalg.norm(X, axis=1), 1.0, atol=1e-4)
    return X, y

X_train, y_train = extract_chunked('train', train_df)
X_val, y_val = extract_chunked('val', val_df)
X_test, y_test = extract_chunked('test', test_df)

np.save(CHUNKS_DIR / 'X_val.npy', X_val)
np.save(CHUNKS_DIR / 'y_val.npy', y_val)
np.save(CHUNKS_DIR / 'X_test.npy', X_test)
np.save(CHUNKS_DIR / 'y_test.npy', y_test)
print('Shapes -> Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)


## 5. Build Versioned `IndexFlatIP` + Post-Embedding Near-Duplicate Leakage Check


In [ ]:
# Build exact normalized IndexFlatIP strictly from TRAINING vectors
full_index = faiss.IndexFlatIP(768)
full_index.add(X_train)

assert full_index.d == 768
assert full_index.ntotal == len(train_df) == len(y_train)
assert full_index.metric_type == faiss.METRIC_INNER_PRODUCT
assert y_train.min() == 0 and y_train.max() == 66

# Deterministic self-consistency check on vector 0
s0, i0 = full_index.search(X_train[0:1], 1)
assert int(i0[0, 0]) == 0 and abs(float(s0[0, 0]) - 1.0) < 1e-4

# Post-embedding cosine leakage audit across Train -> Val & Test
val_top1_s, val_top1_i = full_index.search(X_val, 1)
test_top1_s, test_top1_i = full_index.search(X_test, 1)
leak_audit = {
    'val_to_train_cos_ge_0_99': int((val_top1_s[:, 0] >= 0.99).sum()),
    'val_to_train_cos_ge_0_98': int((val_top1_s[:, 0] >= 0.98).sum()),
    'test_to_train_cos_ge_0_99': int((test_top1_s[:, 0] >= 0.99).sum()),
    'test_to_train_cos_ge_0_98': int((test_top1_s[:, 0] >= 0.98).sum()),
}
# Record top-20 visually closest val->train collisions for human review
top_coll = np.argsort(val_top1_s[:, 0])[::-1][:20]
coll_rows = [
    {
        'val_path': val_df.iloc[int(idx)]['path'],
        'val_label': val_df.iloc[int(idx)]['label'],
        'train_path': train_df.iloc[int(val_top1_i[idx, 0])]['path'],
        'train_label': train_df.iloc[int(val_top1_i[idx, 0])]['label'],
        'cosine': round(float(val_top1_s[idx, 0]), 5),
    }
    for idx in top_coll
]
pd.DataFrame(coll_rows).to_csv(EVAL_OUT / 'val_to_train_nearest_visual_pairs.csv', index=False)
(EVAL_OUT / 'vector_leakage_audit.json').write_text(json.dumps(leak_audit, indent=2))
print('Vector Leakage Audit:', json.dumps(leak_audit, indent=2))

# Save versioned index & full metadata (NOT touching v1 production folder!)
index_path = ASSETS_OUT / 'crop_index_v3_full.faiss'
faiss.write_index(full_index, str(index_path))
faiss.write_index(full_index, str(ASSETS_OUT / 'crop_index.faiss'))  # Canonical filename inside versioned dir
np.save(ASSETS_OUT / 'train_labels.npy', y_train)
train_mapping = train_df[['image_id', 'path', 'label', 'crop', 'class_id', 'sha256', 'dhash']].copy()
train_mapping.insert(0, 'faiss_row_id', np.arange(len(train_mapping)))
train_mapping.to_csv(ASSETS_OUT / 'train_image_mapping.csv', index=False)
print('Saved FAISS index:', round(index_path.stat().st_size / (1024**2), 2), 'MB | Vectors:', full_index.ntotal)


## 6. Compare All 7 Classifier Candidates on Validation Split (Plus Baseline v1 Comparison on Same Split)

Evaluates **on `X_val` strictly**:
1. Old 2,144-reference FAISS (`v1`, if uploaded to `V1_ASSETS_DIR`)
2. Full FAISS `k = 1`
3. Full FAISS `k in [3, 5, 7, 11, 21]` (Majority voting)
4. Full FAISS `k in [3, 5, 7, 11, 21]` (Similarity-weighted voting)
5. Class-balanced retrieval (Inverse class-frequency weighted k-NN + Class-balanced prototype centroids)
6. Crop-aware retrieval (Predicted-crop hierarchical reranking + oracle bound)
7. Linear classifier (`LogisticRegression` on frozen 768-D features)
8. SVM classifier (`LinearSVC` + probability calibration on frozen 768-D features)


In [ ]:
MAX_K = 21
val_sims, val_ids = full_index.search(X_val, MAX_K)
class_counts = np.bincount(y_train, minlength=67).astype(np.float64)
inv_freq = np.median(class_counts) / np.maximum(class_counts, 1.0)
crop_names = sorted(list({c.split('/')[0] for c in CANONICAL_CLASSES}))
cls_to_crop_id = np.array([crop_names.index(c.split('/')[0]) for c in CANONICAL_CLASSES])

def eval_topk_hit(yt, rank_lists, k):
    hits = 0
    for truth, row in zip(yt, rank_lists):
        if int(truth) in list(dict.fromkeys(row))[:k]:
            hits += 1
    return float(hits / len(yt))

def score_scores_matrix(name, yt, score_mat, latency_ms, mem_mb, file_mb):
    top1 = np.argmax(score_mat, axis=1)
    top5_ranks = np.argsort(-score_mat, axis=1)[:, :5]
    top3 = np.mean([yt[i] in top5_ranks[i, :3] for i in range(len(yt))])
    top5 = np.mean([yt[i] in top5_ranks[i, :5] for i in range(len(yt))])
    return {
        'model_name': name,
        'val_top1_acc': round(float(accuracy_score(yt, top1)), 5),
        'val_macro_f1': round(float(f1_score(yt, top1, average='macro', zero_division=0)), 5),
        'val_top3_recall': round(float(top3), 5),
        'val_top5_recall': round(float(top5), 5),
        'latency_ms_per_query': round(latency_ms, 4),
        'index_or_model_ram_mb': round(mem_mb, 2),
        'artifact_size_mb': round(file_mb, 2),
    }, top1

def build_knn_score_matrix(sims, ids, k, mode='weighted'):
    out = np.zeros((len(sims), 67), dtype=np.float64)
    for r, (ss, ii) in enumerate(zip(sims[:, :k], ids[:, :k])):
        ok = ii >= 0
        nbr_y = y_train[ii[ok]]
        nbr_s = ss[ok]
        if mode == 'majority':
            w = np.ones(len(nbr_y), dtype=np.float64)
        elif mode == 'weighted':
            w = np.exp(10.0 * (nbr_s - nbr_s.max()))
        elif mode == 'class_balanced':
            w = np.exp(10.0 * (nbr_s - nbr_s.max())) * inv_freq[nbr_y]
        out[r] = np.bincount(nbr_y, weights=w, minlength=67)
    return out

benchmarks = []
index_mb = (ASSETS_OUT / 'crop_index.faiss').stat().st_size / (1024 ** 2)

# Baseline v1 (2,144 vectors) evaluated on EXACT SAME X_val
v1_index_file = V1_ASSETS_DIR / 'crop_index.faiss'
v1_labels_file = V1_ASSETS_DIR / 'train_labels.npy'
if v1_index_file.exists() and v1_labels_file.exists():
    v1_idx = faiss.read_index(str(v1_index_file))
    v1_y = np.load(v1_labels_file).astype(np.int32)
    t0 = time.perf_counter()
    v1_s, v1_i = v1_idx.search(X_val, 10)
    lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
    s_mat_v1 = np.zeros((len(X_val), 67), dtype=np.float64)
    for r in range(len(X_val)):
        # k=1 primary + top rank list
        top_c = v1_y[v1_i[r, 0]]
        s_mat_v1[r, top_c] += 100.0
        for rank_idx, nb in enumerate(v1_i[r]):
            s_mat_v1[r, v1_y[nb]] += 1.0 / (rank_idx + 10.0)
    row_v1, _ = score_scores_matrix('v1_baseline_2144_k1', y_val, s_mat_v1, lat, v1_index_file.stat().st_size/(1024**2), v1_index_file.stat().st_size/(1024**2))
    benchmarks.append(row_v1)

# 1 & 2 & 3 & 4: Full FAISS sweeps across k & voting schemes
for k in [1, 3, 5, 7, 11, 21]:
    modes = ['majority'] if k == 1 else ['majority', 'weighted', 'class_balanced']
    for mode in modes:
        t0 = time.perf_counter()
        s, ii = full_index.search(X_val, k)
        smat = build_knn_score_matrix(s, ii, k, mode)
        lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
        row, _ = score_scores_matrix(f'full_faiss_k{k}_{mode}', y_val, smat, lat, index_mb, index_mb)
        benchmarks.append(row)

# 5a: Nearest Class Prototypes
t0 = time.perf_counter()
centroids = np.vstack([X_train[y_train == c].mean(axis=0) for c in range(67)]).astype('float32')
faiss.normalize_L2(centroids)
cent_scores = X_val @ centroids.T
lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
row_cent, _ = score_scores_matrix('class_prototypes_cosine', y_val, cent_scores, lat, 0.2, 0.2)
benchmarks.append(row_cent)

# 5b: Hierarchical Crop-Aware Retrieval (Predict crop via crop-level LogReg, then k-NN weighted inside predicted crop)
crop_clf = LogisticRegression(max_iter=1000, C=5.0, n_jobs=-1).fit(X_train, cls_to_crop_id[y_train])
for k_ca in [5, 11, 21]:
    t0 = time.perf_counter()
    pred_crops = crop_clf.predict(X_val)
    raw_smat = build_knn_score_matrix(val_sims, val_ids, k_ca, 'weighted')
    crop_mask = (cls_to_crop_id[None, :] == pred_crops[:, None]).astype(np.float64)
    masked_smat = raw_smat * crop_mask + 1e-9 * raw_smat
    lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
    row_ca, _ = score_scores_matrix(f'crop_aware_2stage_k{k_ca}', y_val, masked_smat, lat, index_mb + 1.0, index_mb + 1.0)
    benchmarks.append(row_ca)

# 6: Linear Classifier (Logistic Regression on frozen 768-D embeddings)
for C_lr in [1.0, 5.0, 10.0]:
    lr = LogisticRegression(max_iter=1000, C=C_lr, class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
    t0 = time.perf_counter()
    probs_lr = lr.predict_proba(X_val)
    lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
    row_lr, _ = score_scores_matrix(f'linear_logreg_balanced_C{C_lr}', y_val, probs_lr, lat, 0.5, 0.5)
    benchmarks.append(row_lr)

# 7: Linear SVM (+ Platt Calibration) on frozen 768-D embeddings
for C_svm in [0.5, 1.0, 2.0]:
    base_svm = LinearSVC(C=C_svm, class_weight='balanced', dual='auto')
    cal_svm = CalibratedClassifierCV(base_svm, cv=3).fit(X_train, y_train)
    t0 = time.perf_counter()
    probs_svm = cal_svm.predict_proba(X_val)
    lat = (time.perf_counter() - t0) * 1000.0 / len(X_val)
    row_svm, _ = score_scores_matrix(f'svm_linear_cal_C{C_svm}', y_val, probs_svm, lat, 1.5, 1.5)
    benchmarks.append(row_svm)

bench_df = pd.DataFrame(benchmarks).sort_values(['val_macro_f1', 'val_top1_acc'], ascending=False)
bench_df.to_csv(EVAL_OUT / 'task3_validation_benchmark.csv', index=False)
print('=== Validation Split Benchmark (All 7 Families) ===')
print(bench_df.to_string(index=False))


## 7. Evaluate Selected Winner vs Old 2,144 Baseline ONCE on Untouched Test Split

**Strictly one test pass** after validation selection. Outputs `test_confusion_matrix_winner.csv`, `test_per_class_winner.csv`, worst-performing classes, and side-by-side comparison against v1 on the identical `X_test`.


In [ ]:
# Select best candidate by validation Macro-F1 (excluding v1 baseline)
candidates_only = bench_df[bench_df.model_name != 'v1_baseline_2144_k1']
WINNER_NAME = str(candidates_only.iloc[0]['model_name'])
print('Selected Winner on Validation:', WINNER_NAME)

import pickle
# Fit or configure the winning predictor cleanly and serialize experimental model (versioned!)
best_lr = LogisticRegression(max_iter=1000, C=5.0, class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
best_svm = CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', dual='auto'), cv=3).fit(X_train, y_train)
with open(ASSETS_OUT / 'experimental_heads.pkl', 'wb') as f:
    pickle.dump({'logreg_c5': best_lr, 'svm_cal_c1': best_svm, 'centroids': centroids, 'winner_val_name': WINNER_NAME}, f)

# Helper to score X_test with any chosen winner cleanly
def score_model_on_split(model_name, X_q, test_sims_full, test_ids_full):
    if model_name.startswith('linear_logreg'):
        c_val = float(model_name.split('_C')[-1])
        clf = LogisticRegression(max_iter=1000, C=c_val, class_weight='balanced', n_jobs=-1).fit(X_train, y_train)
        return clf.predict_proba(X_q)
    elif model_name.startswith('svm_linear'):
        c_val = float(model_name.split('_C')[-1])
        clf = CalibratedClassifierCV(LinearSVC(C=c_val, class_weight='balanced', dual='auto'), cv=3).fit(X_train, y_train)
        return clf.predict_proba(X_q)
    elif model_name == 'class_prototypes_cosine':
        return X_q @ centroids.T
    elif model_name.startswith('crop_aware'):
        k_val = int(model_name.split('_k')[-1])
        pc = crop_clf.predict(X_q)
        raw = build_knn_score_matrix(test_sims_full, test_ids_full, k_val, 'weighted')
        mask = (cls_to_crop_id[None, :] == pc[:, None]).astype(np.float64)
        return raw * mask + 1e-9 * raw
    else:
        parts = model_name.split('_')
        k_val = int([p[1:] for p in parts if p.startswith('k')][0])
        mode = 'class_balanced' if 'balanced' in model_name else ('majority' if 'majority' in model_name else 'weighted')
        return build_knn_score_matrix(test_sims_full, test_ids_full, k_val, mode)

t0 = time.perf_counter()
test_sims_all, test_ids_all = full_index.search(X_test, MAX_K)
win_scores_test = score_model_on_split(WINNER_NAME, X_test, test_sims_all, test_ids_all)
win_lat = (time.perf_counter() - t0) * 1000.0 / len(X_test)
win_pred_test = np.argmax(win_scores_test, axis=1)
win_top5_test = np.argsort(-win_scores_test, axis=1)[:, :5]

test_summary = {
    'winner_name': WINNER_NAME,
    'n_test': int(len(y_test)),
    'test_top1_acc': round(float(accuracy_score(y_test, win_pred_test)), 5),
    'test_macro_f1': round(float(f1_score(y_test, win_pred_test, average='macro', zero_division=0)), 5),
    'test_top3_recall': round(float(np.mean([y_test[i] in win_top5_test[i, :3] for i in range(len(y_test))])), 5),
    'test_top5_recall': round(float(np.mean([y_test[i] in win_top5_test[i, :5] for i in range(len(y_test))])), 5),
    'test_latency_ms': round(win_lat, 3),
}

if v1_index_file.exists():
    v1_ts, v1_ti = v1_idx.search(X_test, 5)
    v1_pred_test = v1_y[v1_ti[:, 0]]
    test_summary['v1_baseline_on_same_test_top1'] = round(float(accuracy_score(y_test, v1_pred_test)), 5)
    test_summary['v1_baseline_on_same_test_macro_f1'] = round(float(f1_score(y_test, v1_pred_test, average='macro', zero_division=0)), 5)

print('=== Final Held-Out Test Comparison ===')
print(json.dumps(test_summary, indent=2))
(EVAL_OUT / 'test_final_comparison.json').write_text(json.dumps(test_summary, indent=2))

cm_test = confusion_matrix(y_test, win_pred_test, labels=np.arange(67))
pd.DataFrame(cm_test, index=CANONICAL_CLASSES, columns=CANONICAL_CLASSES).to_csv(EVAL_OUT / 'test_confusion_matrix_winner.csv')
rep_test = classification_report(y_test, win_pred_test, labels=np.arange(67), target_names=CANONICAL_CLASSES, output_dict=True, zero_division=0)
rep_df = pd.DataFrame(rep_test).T
rep_df.to_csv(EVAL_OUT / 'test_per_class_winner.csv')
print('
Worst 10 Diseases on Held-Out Test:')
print(rep_df.iloc[:67].sort_values('f1-score').head(10).to_string())


## 8. Uncertainty & Out-of-Distribution (OOD) Rejection Calibration

Evaluates on validation-derived stress conditions + held-out OOD categories:
- Clean healthy & diseased validation leaves (in-distribution target to ACCEPT)
- Blurred leaves (`GaussianBlur radius=3, 5` -> image quality / blur gate `blur_laplacian_var`)
- Poor-lighting leaves (`Brightness factor=0.25`)
- Unsupported crops (if `unsupported_crops_manifest.csv` has extra crops, plus Leave-One-Crop-Out simulation)
- Non-leaf photographs (CIFAR-10 animals/vehicles/ships/trucks + human/indoor synthetic + Drive `ood_images/` if provided)
- Ambiguous disease photos (small cosine margin between top-1 and top-2 classes)

Calibrates rejection policy on `val` reporting False-Accept Rate (FAR) and False-Reject Rate (FRR).


In [ ]:
# Extract features for OOD / quality stress tests cleanly without altering FAISS index
def embed_pil_batch(pil_images):
    tensors = torch.stack([PREPROCESS(im.convert('RGB')) for im in pil_images])
    with torch.inference_mode():
        with torch.autocast('cuda', dtype=torch.float16):
            emb = model(tensors.to(DEVICE)).float().cpu().numpy()
    emb /= np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-9)
    return emb.astype('float32')

def laplacian_var_pil(im):
    gray = np.asarray(im.convert('L').resize((256, 256)), dtype=np.uint8)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var()), float(gray.mean())

# Sample up to 300 clean val photos (balanced healthy + diseased), blur & darken copies, plus CIFAR-10 animals/machinery/non-plant
val_sample = val_df.sample(n=min(300, len(val_df)), random_state=42)
clean_ims, blur_ims, dark_ims = [], [], []
clean_q, blur_q, dark_q = [], [], []
for rel in val_sample['path']:
    with Image.open(CROPS_DIR / rel) as raw:
        im = ImageOps.exif_transpose(raw).convert('RGB')
        clean_ims.append(im.copy())
        clean_q.append(laplacian_var_pil(im))
        b_im = im.filter(ImageFilter.GaussianBlur(radius=4.5))
        blur_ims.append(b_im)
        blur_q.append(laplacian_var_pil(b_im))
        d_im = ImageEnhance.Brightness(im).enhance(0.20)
        dark_ims.append(d_im)
        dark_q.append(laplacian_var_pil(d_im))

# Natural Non-Leaf Photos: CIFAR-10 objects/animals (horse, deer, truck, automobile, frog, ship, cat, dog)
cifar_ood = tv_datasets.CIFAR10(root='/content/cifar_ood', train=False, download=True)
non_leaf_ims = [cifar_ood[i][0].resize((256, 256), Image.Resampling.BILINEAR) for i in range(300)]
non_leaf_q = [laplacian_var_pil(im) for im in non_leaf_ims]

X_clean = embed_pil_batch(clean_ims)
X_blur = embed_pil_batch(blur_ims)
X_dark = embed_pil_batch(dark_ims)
X_nonleaf = embed_pil_batch(non_leaf_ims)

def ood_signals(X_q):
    sims, ids = full_index.search(X_q, 5)
    top1_sim = sims[:, 0]
    nbr_c = y_train[ids]
    agree = np.array([
        float((np.bincount(row, weights=np.exp(10.0*(sc - sc.max())), minlength=67).max()) /
              max(np.bincount(row, weights=np.exp(10.0*(sc - sc.max())), minlength=67).sum(), 1e-9))
        for row, sc in zip(nbr_c, sims)
    ])
    lr_probs = best_lr.predict_proba(X_q)
    top2_sorted = np.sort(lr_probs, axis=1)[:, -2:]
    prob_max = top2_sorted[:, 1]
    prob_margin = top2_sorted[:, 1] - top2_sorted[:, 0]
    return top1_sim, agree, prob_max, prob_margin

sig_clean = ood_signals(X_clean)
sig_blur = ood_signals(X_blur)
sig_dark = ood_signals(X_dark)
sig_nonleaf = ood_signals(X_nonleaf)

# Optional: Unsupported crops from archive if present
sig_unsup = None
if UNSUPPORTED_CSV.exists():
    udf = pd.read_csv(UNSUPPORTED_CSV)
    if len(udf) > 0:
        u_ims = [ImageOps.exif_transpose(Image.open(CROPS_DIR / p)).convert('RGB') for p in udf['path'].head(300)]
        X_unsup = embed_pil_batch(u_ims)
        sig_unsup = ood_signals(X_unsup)

# Threshold Sweep for explicit rejection policy on Validation Data
sweep_rows = []
for th_sim in [0.45, 0.50, 0.55, 0.60, 0.65]:
    for th_prob in [0.30, 0.45, 0.60]:
        for th_blur in [0.0, 40.0, 75.0]:
            def accepted(sig, qual):
                s, a, pm, margin = sig
                b_var = np.array([q[0] for q in qual])
                bright = np.array([q[1] for q in qual])
                return (s >= th_sim) & (pm >= th_prob) & (b_var >= th_blur) & (bright >= 25.0)
            frr_clean = float((~accepted(sig_clean, clean_q)).mean())
            far_nonleaf = float(accepted(sig_nonleaf, non_leaf_q).mean())
            far_blur = float(accepted(sig_blur, blur_q).mean())
            far_dark = float(accepted(sig_dark, dark_q).mean())
            row_sw = {
                'th_sim': th_sim, 'th_prob': th_prob, 'th_blur': th_blur,
                'FRR_clean_leaves': round(frr_clean, 4),
                'FAR_non_leaf': round(far_nonleaf, 4),
                'FAR_blurry': round(far_blur, 4),
                'FAR_dark': round(far_dark, 4),
            }
            if sig_unsup is not None:
                row_sw['FAR_unsupported_crop'] = round(float(accepted(sig_unsup, [(200, 128)]*len(sig_unsup[0])).mean()), 4)
            sweep_rows.append(row_sw)

sweep_df = pd.DataFrame(sweep_rows).sort_values(['FAR_non_leaf', 'FRR_clean_leaves'])
sweep_df.to_csv(EVAL_OUT / 'ood_threshold_calibration_sweep.csv', index=False)
print('Top calibrated rejection policies (lowest Non-Leaf FAR + Clean FRR):')
print(sweep_df.head(15).to_string(index=False))
